### feature = 문제 / 입력 데이터 / 모델이 보는 정보
### target  = 정답 / 예측해야 하는 값

## 하이퍼파라미터 기본 개념
하이퍼파라미터: 모델이 데이터에서 스스로 학습하는 값이 아니라, 사람이 학습 전에 미리 정하는 설정값임.

- alpha: 규제를 얼마나 강하게 적용할지 정하는 값임.
- l1_ratio: ElasticNet에서 L1 규제와 L2 규제를 어떤 비율로 섞을지 정하는 값임.

# Regularized Linear Models 규제 선형 회귀
다항식이 복잡해지져서 회귀계수가 매우 크게 설정이되면서 과대적합이 되고
평가데이터세트에 대해서 형편없는 예측 성능을 보이게 된다.

비용함수의 최소값을 구하는 것이 모델이 추구하는 목적이므로,
비용함수의 최소값을 구하는데 **𝝰 제약**을 걸어 과적합을 방지하는 것을 규제라고 한다.
- Lasso L1방식의 규제 적용
- Ridge L2방식의 규제 적용
- ElasticNet L1, L2규제를 결합한 모델. 특성이 많은 데이터셋에 적용. L1규제로 특성개수를 줄이고, L2규제로 계수값의 크기도 조정할 수 있다.


**비용함수 목표**

$
비용함수 목표 = \min \left(\text{RSS}(w) + \alpha \times W\right)
$

이 수식은 규제를 적용한 비용함수를 의미하며, 다음과 같은 요소들로 이루어져 있다:
1. **RSS(w)**: 이 부분은 Residual Sum of Squares의 약자로, 잔차 제곱합을 의미한다. 선형 회귀 모델에서 주로 사용하는 손실 함수로, 각 데이터 포인트에서의 예측 값과 실제 값 간의 차이를 제곱한 것들의 합이다. 즉, 모델의 예측 오차를 측정하는 부분이다. 수식으로는 다음과 같이 표현된다:
    $
    \text{RSS}(w) = \sum_{i=1}^n \left(y_i - \hat{y}_i\right)^2
    $
   여기서 $y_i$는 실제 값, $\hat{y}_i$는 모델의 예측 값이다.
2. **$\alpha$**: 규제 강도를 나타내는 하이퍼파라미터이다. 이 값이 클수록 규제의 효과가 커지고, 작을수록 규제의 효과가 줄어든다. 모델이 과적합되기 쉬운 경우, $\alpha$를 크게 설정하여 가중치를 제어할 수 있다.
3. **$W$**: 가중치들의 규제 항을 의미한다. 이는 가중치의 크기에 페널티를 부여하는 부분으로, 모델의 복잡도를 조절하는 역할을 한다.
   - L1 규제: $W = \sum_{j=1}^p |w_j|$
   - L2 규제: $W = \sum_{j=1}^p w_j^2$
수식을 다시 설명하면, 비용함수의 목표는 잔차 제곱합(RSS)과 규제 항($\alpha \times W$)을 더한 값을 최소화하는 것이다. 즉, 이 비용함수의 최적화 목표는 두 가지를 달성하고자 한다:
1. 모델의 예측 오차(RSS)를 줄이는 것.
2. 모델의 복잡도를 줄여서 가중치의 크기를 제어하는 것($\alpha \times W$).

**적합합 규제를 선택하려면 :**

1. **Lasso (L1 규제)**
  - 불필요한 피처를 자동으로 제거할 때 유용하다.
  - 많은 피처 중 일부만 중요할 때 사용하면 스파스한 모델을 만듦.
2. **Ridge (L2 규제)**
  - 모든 피처가 유의미하고 예측에 기여한다고 생각될 때 적합하다.
  - 피처가 많고 과적합을 방지하고 싶을 때 사용한다.
3. **Elastic Net (L1 + L2 규제)**
  - 피처를 일부 제거하면서도 나머지의 가중치도 줄이고 싶을 때 유용하다.
  - 상관관계가 높은 피처가 있을 때 선택을 안정적으로 한다.


## L2
- L2방식의 규제를 구현한 Ridge 클래스를 사용할 수 있다.
- 모든 피쳐의 회귀계수를 규제해 과적합을 방지한다.


## 실습 환경 준비

- NumPy: 배열 계산과 수치 연산을 다루기 위한 기본 라이브러리임.
- Pandas: 표 형태 데이터를 DataFrame으로 다루기 위한 라이브러리임.
- Matplotlib: 그래프를 그려 데이터 분포와 모델 결과를 시각화하는 라이브러리임.
- Seaborn: 통계 그래프를 더 쉽게 그리기 위한 시각화 라이브러리임.


In [1]:
# 초기 세팅용 import 구문입니다. 먼저 실행한 뒤 실습 코드를 작성합니다.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.dates import drange
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import mean_squared_error, mean_absolute_error, root_mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet


## California Housing 데이터 불러오기

In [6]:
# 스켈레튼에 켈리포니아 하우징 데이터 불러오기
from sklearn.datasets import fetch_california_housing

california_housing = fetch_california_housing()

# 학습할 입력값(X : X_train, X_test 등등 변경)
california_housing_df = pd.DataFrame(  # 데이터 프레임으로 변경 후
    california_housing.data,  # 데이터 입력
    columns=california_housing.feature_names  # 컬럽값을 지정한다
)

# print(california_housing.target)

# 정답(y) - 예측해야할 주택의 가격 딕셔너리 추가
california_housing_df['MedHouseVal'] = california_housing.target
california_housing_df.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25


## 학습/평가 데이터 분리

- train_test_split: 데이터를 학습용과 평가용으로 나누어 새 데이터 성능을 확인할 준비를 함.


In [7]:
# train_test_split() : 데이터의 일부를 떄서 확인한다
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    california_housing.data,
    california_housing.target,
    test_size=0.2,
    random_state=42
)

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

(16512, 8) (16512,)
(4128, 8) (4128,)


## 회귀 평가지표
점수가 낮을수록 좋음
- MSE: 오차를 제곱해 평균낸 값으로 큰 오차에 더 민감함.
- MAE: 오차의 절댓값을 평균낸 값으로 실제 단위 해석이 쉬움.
- RMSE: MSE에 제곱근을 씌워 target과 같은 단위로 해석하는 지표임.


---

점수가 높을수록 좋음
- fit: 훈련 데이터에서 모델 또는 전처리 기준을 학습하는 메서드임.
- predict: 학습된 모델로 새 데이터의 예측값을 생성하는 메서드임.
- score: 모델의 기본 평가 점수를 계산하는 메서드임.


# 다항 feature(다항 선형) - LinearRegression 모델 평가 지표


In [15]:
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import mean_squared_error, mean_absolute_error, root_mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

# 파이프라인이란?
# 모댈생성, 학습, 변환 작업물들을 순서대로 진행하게하는 모델(메소드)
pipeline = Pipeline([
    # 1. 기존 feature를 2차 다항 feature로 확장한다.
    # 예: length, height, width가 있으면
    # length^2, length height, height^2 같은 새로운 feature가 만들어진다.
    # include_bias=False는 상수항 1을 추가하지 않겠다는 의미이다.
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),

    # 2. feature들의 평균과 표준편차를 기준으로 값을 표준화한다.
    # 평균은 0, 표준편차는 1에 가깝게 맞춘다.
    # 다항 feature처럼 값의 크기가 달라질 수 있는 경우 스케일링이 중요하다.
    ('scaler', StandardScaler()),

    # 3. 변환된 feature를 사용해서 선형 회귀 모델을 학습한다.
    # fit_intercept=True는 절편을 학습하겠다는 의미이다.
    ('linear_regression', LinearRegression(fit_intercept=True)),
])

## 학습진행 -> 예측 -> 평가
# 학습 진행
# 파이프라인 전체를 학습한다.
# X_train: 입력 feature, y_train: 정답 target
# 내부적으로 다항 feature 생성 -> 스케일링 -> 선형 회귀 학습 순서로 진행된다.
pipeline.fit(X_train, y_train)

# 학습된 선형 회귀 모델만 따로 확인하고 싶을 때 꺼낸다.
model = pipeline.named_steps['linear_regression']

# 학습 데이터에 대한 예측값
# 파이프라인 내부 변환 과정을 자동으로 거친 뒤 예측한다.
y_train_pred = pipeline.predict(X_train)

# 테스트 데이터에 대한 예측값
# fit은 다시 하지 않고, 학습 때 정한 변환 기준으로 예측한다.
y_test_pred = pipeline.predict(X_test)

# 회귀 모델의 평가 결과를 표 형태로 정리한다.
linear_eval_result = pd.DataFrame({
    # dataset: 평가 대상 데이터가 train인지 test인지 구분하는 컬럼
    'dataset': ['train', 'test'],

    # R2: 결정계수
    # 모델이 정답 y를 얼마나 잘 설명하는지 나타내는 점수
    # 1에 가까울수록 좋고, 0에 가까우면 평균으로 예측하는 것과 비슷하다.
    'R2': [
        pipeline.score(X_train, y_train),
        pipeline.score(X_test, y_test)
    ],

    # MSE: Mean Squared Error, 평균 제곱 오차
    # 실제값과 예측값의 차이를 제곱한 뒤 평균낸 값
    # 오차를 제곱하기 때문에 큰 오차에 더 민감하다.
    # 작을수록 좋다.
    'MSE': [
        mean_squared_error(y_train, y_train_pred),
        mean_squared_error(y_test, y_test_pred)
    ],

    # MAE: Mean Absolute Error, 평균 절대 오차
    # 실제값과 예측값의 차이를 절댓값으로 바꾼 뒤 평균낸 값
    # 예측값이 실제값과 평균적으로 얼마나 차이 나는지 직관적으로 볼 수 있다.
    # 작을수록 좋다.
    'MAE': [
        mean_absolute_error(y_train, y_train_pred),
        mean_absolute_error(y_test, y_test_pred)
    ],

    # RMSE: Root Mean Squared Error, 평균 제곱근 오차
    # MSE에 루트를 씌운 값
    # target과 같은 단위로 해석할 수 있다.
    # 작을수록 좋다.
    'RMSE': [
        root_mean_squared_error(y_train, y_train_pred),
        root_mean_squared_error(y_test, y_test_pred)
    ]
})

# model.coef_는 학습된 선형 회귀 모델의 회귀계수이다.
# 각 feature가 예측값에 얼마나 영향을 주는지 나타내는 가중치이다.
# 회귀계수 제곱합은 계수들이 전체적으로 얼마나 큰지 확인하는 값이다.
print('회귀계수 제곱합:', np.sum(model.coef_ ** 2))

linear_eval_result

회귀계수 제곱합: 7170.956218225917


,dataset,R2,MSE,MAE,RMSE
0,train,0.685268,0.420727,0.460838,0.648634
1,test,0.645682,0.464302,0.467001,0.681397


### 최적의 alpha값 찾기


## 교차검증

- cross_val_score: 여러 fold의 검증 점수를 계산해 평균 성능을 더 안정적으로 확인함.


## L1
- L1규제방식을 구현한 Lasso클래스를 사용할 수 있다.
- alpha값을 통해 특정 회귀계수를 0까지 제한, 특정속성을 회귀계산에서 배제하는 것도 가능.


## 회귀 평가지표

- MSE: 오차를 제곱해 평균낸 값으로 큰 오차에 더 민감함.
- fit: 훈련 데이터에서 모델 또는 전처리 기준을 학습하는 메서드임.
- predict: 학습된 모델로 새 데이터의 예측값을 생성하는 메서드임.


## L1 + L2
L1규제, L2규제를 적절한 비율로 모두 적용하는 ElasticNet 선형회귀모델을 사용할 수 있다.
ElasticNet은 **회귀 분석** 기법 중 하나로, **Lasso**와 **Ridge**의 규제를 결합한 모델이다.
Lasso는 특성 선택에 효과적이고, Ridge는 모든 특성을 다루면서 모델을 규제한다.
ElasticNet은 이 두 가지 규제(L1과 L2)를 적절히 혼합하여 사용하는 방법이다.
**alpha 파라미터**
alpha는 a + b를 의미한다.
- a는 L1규제용 alpha값이다.
- b는 L2규제용 alpha값이다.
**l1_ratio 파라미터**
L1규제용 alpha값의 비율이다. $\frac{a}{a + b}$
- alpha가 10이고, l1_ratio가 0.7이면 a = 7, b = 3이다.
- alpha가 10이고, l1_ratio가 1이면 a = 10, b = 0이다. 즉, L1규제만 사용한다.
- alpha가 10이고, l1_ratio가 0이면 a = 0, b = 10이다. 즉, L2규제만 사용한다.
**수식:**
$$J(β) = RSS + α [ λ * ||β||₁ + (1 - λ) * ||β||₂² ]$$
- **RSS**: Residual Sum of Squares (예측 오차)
- **α**: 전체 규제 강도 (크면 규제가 강해짐)
- **λ**: L1과 L2 규제의 비율 조절 (0 ≤ λ ≤ 1)
  - λ = 1 → Lasso만 적용
  - λ = 0 → Ridge만 적용
- **||β||₁**: L1 노름 (∑|βᵢ|), 특성 선택
- **||β||₂²**: L2 노름 제곱 (∑βᵢ²), 계수 축소
**특징:**
- **Lasso와 Ridge의 장점을 결합**: ElasticNet은 Lasso의 **특성 선택** 능력과 Ridge의 **강한 규제** 특성을 모두 반영한다.
- **고차원 데이터에 적합**: 상관관계가 높은 특성이 많은 데이터나 차원이 높은 데이터에 적합하다.
- **Overfitting 방지**: 두 가지 규제를 혼합하여 과적합을 효과적으로 방지할 수 있다.


## 모델 학습

- fit: 훈련 데이터에서 모델 또는 전처리 기준을 학습하는 메서드임.
- score: 모델의 기본 평가 점수를 계산하는 메서드임.


## 결과 확인

- 실행 결과: 앞에서 만든 객체와 실행 결과를 확인하며 다음 단계로 연결함.


## 다중공선성 MultiCollinearity
특성간의 상관관계가 너무 높은 경우를 가리킨다.
주택데이터에서 면적특성과 방의크기특성은 높은 상관관계(상관계수 0.8이상)를 가질수 있다.
다중공선성특성에 대한 회귀계수가 크게 학습이 되고, 이는 특정데이터에 민감한 과대적합을 유발한다.
**해결책**
- 다중공선성 특성 제거
- 규제모델을 사용한 회귀계수 억제


## 결과 확인

- 실행 결과: 앞에서 만든 객체와 실행 결과를 확인하며 다음 단계로 연결함.


## 학습/평가 데이터 분리

- train_test_split: 데이터를 학습용과 평가용으로 나누어 새 데이터 성능을 확인할 준비를 함.
- MSE: 오차를 제곱해 평균낸 값으로 큰 오차에 더 민감함.
- fit: 훈련 데이터에서 모델 또는 전처리 기준을 학습하는 메서드임.
- predict: 학습된 모델로 새 데이터의 예측값을 생성하는 메서드임.
